# GRPO Training — PHI De-identification

Trains the SFT-initialised Qwen3-0.6B **base** model with Group Relative Policy
Optimization (GRPO). Reward is fully computational — no LLM judge, no GPU verifier.

Data: `EdgeDeID/synthetic-dataset` (`train.csv` / `test.csv`), streamed.
Columns: `original` (note) · `anonymized` (ground-truth `<phi_entities>` XML) · `examples`.
The prompt reproduces the **SFT flat-text template** (not a chat template) plus a
mandatory `<qc_status>` self-assessment rule.

**Reward cascade (sequential gate):**
1. `structure`      — valid `<phi_entities>` XML? (hard gate → 0 if it fails)
2. `presence`       — do tagged entities appear verbatim in the source note? (anti-hallucination)
3. `completeness`   — entity-level **F-β (β=2, recall-weighted)** vs ground truth; soft-match has a length guard against 1-char hacks
4. `qc_calibration` — **asymmetric**: confidently claiming `complete` while PHI was missed (leakage) scores 0; over-cautious `incomplete` scores 0.5
5. `length_penalty` — penalise redundant / over-long outputs (token-estimated)

In [8]:
import os
import sys
from pathlib import Path

# Set this one line when moving between local / Kaggle / Colab mounts.
PROJECT_ROOT_HINT = None  # e.g. '/kaggle/input/datasets/sinadaneshgar/edgedeid' or '/content/drive/MyDrive/EdgeDeID'
if PROJECT_ROOT_HINT is None:
    PROJECT_ROOT_HINT = Path.cwd().parent if Path.cwd().name == "RL" else Path.cwd()
PROJECT_ROOT_HINT = Path(PROJECT_ROOT_HINT).expanduser().resolve()
if str(PROJECT_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_HINT))

from pipeline.runtime import detect_runtime_paths

RUNTIME = detect_runtime_paths(project_root=PROJECT_ROOT_HINT)
REPO_ROOT = RUNTIME["project_root"]
OUTPUT_ROOT = RUNTIME["output_root"]
print("runtime:", RUNTIME["env"])
print("REPO_ROOT:", REPO_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

# load gated-HF credentials from .env (HF_TOKEN) — dataset & model are private
env_path = REPO_ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
HF_TOKEN = os.environ.get("HF_TOKEN")
print("HF_TOKEN loaded:", bool(HF_TOKEN))

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from pipeline.rl import GRPOConfig
from pipeline.rl.data import prepare_grpo_split
from pipeline.rl.grpo import build_grpo_trainer, prepare_grpo_reward_function
from pipeline.rl.evaluation import evaluate_policy, compute_metrics
from pipeline.training import expand_tokenizer

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

runtime: local
REPO_ROOT: E:\Sharif\Research\LLM Deidentification\EdgeDeID
OUTPUT_ROOT: E:\Sharif\Research\LLM Deidentification\EdgeDeID
HF_TOKEN loaded: True
torch: 2.8.0+cpu | CUDA: False


## 1 — Configuration

Adjust paths and hyper-parameters here.

In [9]:
# ── model ──────────────────────────────────────────────────────────────────
SFT_MODEL_ID   = "EdgeDeID/Qwen3-0.6B-Base_SFT"      # HF repo of the SFT checkpoint
OUTPUT_DIR     = str(OUTPUT_ROOT / "checkpoints" / "grpo-qwen3-0.6b")

# ── dataset ────────────────────────────────────────────────────────────────
DATASET_ID     = "EdgeDeID/synthetic-dataset"        # HF dataset repo (gated)
TRAIN_FILE     = "train.csv"
EVAL_FILE      = "test.csv"                           # set None to skip eval
NOTE_KEY       = "original"                           # raw note column
GT_KEY         = "anonymized"                         # ground-truth <phi_entities> XML
EXAMPLES_KEY   = "examples"                           # "NONE" or JSON few-shot list

# ── training ───────────────────────────────────────────────────────────────
config = GRPOConfig(
    model_name              = SFT_MODEL_ID,
    output_dir              = OUTPUT_DIR,
    run_name                = "grpo-edgedeid-qwen3-0.6b",
    wandb_project           = None,             # set a name to enable wandb
    learning_rate           = 1e-5,
    per_device_train_batch_size = 1,
    per_device_eval_batch_size  = 1,
    num_generations         = 8,                # completions per prompt per step
    gradient_accumulation_steps = 8,
    max_steps               = 1000,             # required: streaming has no len()
    max_prompt_length       = 2048,
    max_completion_length   = 1024,
    seed                    = 2025,
    reward_weights          = {                 # weights for combined reward
        "presence":     0.30,
        "completeness": 0.40,                   # recall-weighted F-beta (beta=2)
        "qc":           0.15,                   # asymmetric calibration
        "length":       0.15,
    },
)

print(config)
print("OUTPUT_DIR:", OUTPUT_DIR)

GRPOConfig(model_name='EdgeDeID/Qwen3-0.6B-Base_SFT', output_dir='E:\\Sharif\\Research\\LLM Deidentification\\EdgeDeID\\checkpoints\\grpo-qwen3-0.6b', max_prompt_length=2048, max_completion_length=1024, per_device_train_batch_size=1, per_device_eval_batch_size=1, learning_rate=1e-05, seed=2025, run_name='grpo-edgedeid-qwen3-0.6b', tags=[], num_generations=8, group_size=4, max_steps=1000, gradient_accumulation_steps=8, logging_steps=10, reward_clip_min=None, reward_clip_max=None, reward_weights={'presence': 0.3, 'completeness': 0.4, 'qc': 0.15, 'length': 0.15}, wandb_project=None)
OUTPUT_DIR: E:\Sharif\Research\LLM Deidentification\EdgeDeID\checkpoints\grpo-qwen3-0.6b


## 2 — Load Model & Tokenizer

Loads the base model into memory, forcing `float16` to ensure compatibility with GPUs that do not support `bfloat16` natively.
This step will also optionally wrap the model with **PEFT/LoRA** for memory-efficient training if `USE_LORA` is enabled.

In [ ]:
# ── 1. Load Model & Tokenizer ──────────────────────────────────────────────
device_map = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Loading {SFT_MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_ID, token=HF_TOKEN)

model = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_ID,
    torch_dtype=torch.float16,   # Forced to float16 to support older GPUs (like GTX 1660 Ti)
    device_map=device_map,
    token=HF_TOKEN
)
expand_tokenizer(model, tokenizer)
print("Model loaded successfully.")

# ── 2. PEFT / LoRA (Optional) ──────────────────────────────────────────────
USE_LORA = False   # set True on low-VRAM machines

if USE_LORA:
    from peft import LoraConfig, get_peft_model, TaskType
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        target_modules="all-linear",
        lora_dropout=0.05,
        bias="none",
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
else:
    print("Running in full-parameter mode (No LoRA).")

## 3 — Prepare Dataset

Streams the synthetic training dataset directly from the Hugging Face Hub (gated). 
Using `streaming=True` ensures we use an `IterableDataset` so that we do not have to hold the entire dataset in memory. The row mapping lazily formats the input features according to the configured `GRPOConfig`.

In [ ]:
from huggingface_hub import hf_hub_download

# the dataset repo is gated → resolve files with the token, then stream the CSV.
# streaming=True keeps an IterableDataset (no full in-memory materialisation).
train_path = hf_hub_download(
    repo_id=DATASET_ID, filename=TRAIN_FILE, repo_type="dataset", token=HF_TOKEN
)
data_files = {"train": train_path}
if EVAL_FILE:
    data_files["test"] = hf_hub_download(
        repo_id=DATASET_ID, filename=EVAL_FILE, repo_type="dataset", token=HF_TOKEN
    )

raw = load_dataset("csv", data_files=data_files, streaming=True)
print(raw)
print("columns:", next(iter(raw["train"])).keys())

In [ ]:
# map raw rows -> {prompt, source_note, ground_truth}, lazily (stays streaming).
# the GRPO prompt reproduces the SFT flat-text template (+ <qc_status> rule),
# NOT a chat template — the SFT checkpoint is a base model.
train_dataset = prepare_grpo_split(
    raw["train"],
    note_key=NOTE_KEY,
    ground_truth_key=GT_KEY,
    examples_key=EXAMPLES_KEY,
)

eval_dataset = None
if EVAL_FILE and "test" in raw:
    eval_dataset = prepare_grpo_split(
        raw["test"],
        note_key=NOTE_KEY,
        ground_truth_key=GT_KEY,
        examples_key=EXAMPLES_KEY,
    )

# streaming IterableDataset has no len(); peek one record instead.
sample0 = next(iter(train_dataset))
print("keys:", list(sample0.keys()))
print("\n--- prompt (head) ---\n", sample0["prompt"][:600])
print("\n--- ground_truth (head) ---\n", sample0["ground_truth"][:300])

## 4 — Sanity-check Reward Functions

Run the reward pipeline on a single example before launching training.

In [ ]:
from pipeline.rl.rewards import (
    reward_xml_structure,
    reward_phi_text_presence,
    reward_completeness,
    reward_qc_calibration,
    reward_length_penalty,
    compute_grpo_reward,
)

sample = next(iter(train_dataset))   # IterableDataset → no integer indexing
note   = sample["source_note"]
gt     = sample["ground_truth"]

# use the ground truth itself as a perfect-score completion
fake_completion = gt + "\n<qc_status>complete</qc_status>"

c = reward_completeness(fake_completion, gt)
print("structure   :", reward_xml_structure(fake_completion))
print("presence    :", reward_phi_text_presence(fake_completion, note))
print("completeness:", c, "(F-beta, beta=2 recall-weighted)")
print("qc          :", reward_qc_calibration(fake_completion, c), "(asymmetric)")
print("length      :", reward_length_penalty(fake_completion))
print("combined    :", compute_grpo_reward(fake_completion, note, gt))

# negative control: a degenerate 1-char hack should NOT score well
hack = "<phi_entities><names><name id='1' type='patient'><variations>" \
       "<value>a</value></variations></name></names></phi_entities>" \
       "<qc_status>complete</qc_status>"
print("\nhack completeness:", reward_completeness(hack, gt), "(expect ~0, length guard)")

## 5 — Build Trainer & Train

Initialize Weights & Biases (if configured) and build the GRPO Trainer using the compiled reward functions and datasets. The trainer uses `trl` under the hood to manage generation and trajectory-based optimisation.

In [ ]:
import wandb
if config.wandb_project:
    wandb.init(project=config.wandb_project, name=config.run_name)

In [ ]:
reward_fn = prepare_grpo_reward_function(config)

trainer = build_grpo_trainer(
    model        = model,
    tokenizer    = tokenizer,
    config       = config,
    reward_fn    = reward_fn,
    train_dataset= train_dataset,
    eval_dataset = eval_dataset,
)

trainer.train()

## 6 — Evaluate

Runs a quick evaluation pipeline (using temperature inference and token batching) on the offline `eval_dataset` test split to estimate structural fidelity and F-beta scores post-RL.

In [ ]:
if eval_dataset is not None:
    metrics = evaluate_policy(
        model     = trainer.model,
        tokenizer = tokenizer,
        dataset   = eval_dataset,
        max_new_tokens = 1024,
        temperature    = 0.0,
        batch_size     = 4,
    )
    print("Eval metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
else:
    print("No eval split — skipping evaluation.")

## 7 — Save & Push to HuggingFace

Commits the final model weights and tokenizer configurations back to local disk and pushes them to your targeted Hugging Face Hub repository.

In [ ]:
HF_REPO_ID = "EdgeDeID/Qwen3-0.6B-GRPO"   # target HF repo

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

# push to HF Hub
trainer.model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)
print(f"Pushed to {HF_REPO_ID}")

In [ ]:
if config.wandb_project:
    wandb.finish()